# Phase 7 — paired pilot-adapter evaluation

Runs a deterministic 100-record screen before permitting the complete frozen 500-record benchmark. Attach the completed Phase 5 and Phase 6 notebook outputs and use Kaggle GPU T4 x2.

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

RUN_STAGE = "screen"  # Change to "full" only when the screen decision is continue.
BRANCH = "refactoring/phase-transformer"
REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_ROOT = Path("/kaggle/temp/text-summarizer/project")
OUTPUT_DIRECTORY = Path("/kaggle/working/text-summarizer/phase7/pilot-evaluation-v1")

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Stage: {RUN_STAGE}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
requirements = [
    "transformers>=4.57,<4.58",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.48,<1",
    "peft>=0.17,<0.19",
    "bert-score>=0.3.13,<0.4",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("Phase 7 dependencies are ready.")

In [ ]:
if not (PROJECT_ROOT / ".git").is_dir():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
ml_source = str(PROJECT_ROOT / "ml" / "src")
if ml_source not in sys.path:
    sys.path.insert(0, ml_source)
print(f"Project commit: {commit}")

In [ ]:
phase5_candidates = []
for path in sorted(Path("/kaggle/input").rglob("baseline-report.json")):
    parent = path.parent
    if all(
        (parent / name).is_file()
        for name in ("benchmark-selection.jsonl", "base-predictions.jsonl", "bertscore.jsonl")
    ):
        phase5_candidates.append(path)
if len(phase5_candidates) != 1:
    raise RuntimeError(f"Expected exactly one complete Phase 5 input; found {phase5_candidates}")
PHASE5_REPORT_PATH = phase5_candidates[0]
PHASE5_DIRECTORY = PHASE5_REPORT_PATH.parent

pilot_candidates = []
for path in sorted(Path("/kaggle/input").rglob("training-report.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("run_mode") != "pilot":
        continue
    if all(
        (path.parent / relative).is_file()
        for relative in (
            "COMPLETED",
            "adapter/adapter_model.safetensors",
            "adapter/adapter_config.json",
            "tokenizer/tokenizer_config.json",
        )
    ):
        pilot_candidates.append(path)
if len(pilot_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one complete Phase 6 pilot input; found {pilot_candidates}"
    )
PILOT_REPORT_PATH = pilot_candidates[0]
PILOT_DIRECTORY = PILOT_REPORT_PATH.parent
ADAPTER_DIRECTORY = PILOT_DIRECTORY / "adapter"
TOKENIZER_DIRECTORY = PILOT_DIRECTORY / "tokenizer"

print(f"Phase 5 input: {PHASE5_DIRECTORY}")
print(f"Phase 6 pilot: {PILOT_DIRECTORY}")

In [ ]:
from summarizer_ml.evaluation import (
    Phase5Report,
    Phase7Config,
    Phase7Decision,
    Phase7Stage,
    load_benchmark_selection,
    load_predictions,
    select_phase7_examples,
)

ACTIVE_STAGE = Phase7Stage(RUN_STAGE)
CONFIG = Phase7Config()
PHASE5_REPORT = Phase5Report.model_validate_json(PHASE5_REPORT_PATH.read_text(encoding="utf-8"))
SELECTION_PATH = PHASE5_DIRECTORY / "benchmark-selection.jsonl"
BASE_PREDICTIONS_PATH = PHASE5_DIRECTORY / "base-predictions.jsonl"
BASE_BERTSCORE_PATH = PHASE5_DIRECTORY / "bertscore.jsonl"
ADAPTER_PATH = ADAPTER_DIRECTORY / "adapter_model.safetensors"
full_selection = load_benchmark_selection(SELECTION_PATH)
evaluated_selection = select_phase7_examples(full_selection, CONFIG, ACTIVE_STAGE)
base_predictions = load_predictions(BASE_PREDICTIONS_PATH)
if len(full_selection) != PHASE5_REPORT.selected_records or len(base_predictions) != len(
    full_selection
):
    raise RuntimeError("Frozen Phase 5 benchmark coverage is inconsistent.")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
config_path = OUTPUT_DIRECTORY / "phase7-config.json"
config_payload = CONFIG.model_dump_json(indent=2) + "\n"
if config_path.is_file() and config_path.read_text(encoding="utf-8") != config_payload:
    raise RuntimeError("Existing Phase 7 output used a different configuration.")
config_path.write_text(config_payload, encoding="utf-8")
print(CONFIG.model_dump_json(indent=2))
print(f"Frozen benchmark records: {len(full_selection):,}")
print(f"Active {RUN_STAGE} records: {len(evaluated_selection):,}")

In [ ]:
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Phase 7 requires Kaggle GPU T4 x2.")
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {properties.name}; memory={properties.total_memory / (1024**3):.1f} GiB")

In [ ]:
if ACTIVE_STAGE is Phase7Stage.FULL:
    screen_report_path = OUTPUT_DIRECTORY / "screen-comparison-report.json"
    if not screen_report_path.is_file():
        raise RuntimeError("Run the 100-record screen before the full evaluation.")
    screen_report = json.loads(screen_report_path.read_text(encoding="utf-8"))
    if screen_report["decision"] != Phase7Decision.CONTINUE.value:
        raise RuntimeError(
            f"Full evaluation is blocked by screen decision: {screen_report['decision']}"
        )

command = [
    sys.executable,
    "-m",
    "accelerate.commands.launch",
    "--num_processes",
    "2",
    "--num_machines",
    "1",
    "--mixed_precision",
    "fp16",
    "--multi_gpu",
    str(PROJECT_ROOT / "ml" / "scripts" / "generate_adapter_predictions.py"),
    "--phase5-report",
    str(PHASE5_REPORT_PATH),
    "--selection",
    str(SELECTION_PATH),
    "--adapter-directory",
    str(ADAPTER_DIRECTORY),
    "--tokenizer-directory",
    str(TOKENIZER_DIRECTORY),
    "--phase7-config",
    str(config_path),
    "--output-directory",
    str(OUTPUT_DIRECTORY),
    "--stage",
    RUN_STAGE,
]
print("Launching:", " ".join(command))
environment = os.environ.copy()
environment["PYTHONPATH"] = os.pathsep.join(
    value for value in [ml_source, environment.get("PYTHONPATH")] if value
)
subprocess.run(command, check=True, cwd=PROJECT_ROOT, env=environment)

In [ ]:
from summarizer_ml.evaluation import merge_prediction_shards

worker_paths = tuple(OUTPUT_DIRECTORY.glob("workers/rank-*.jsonl"))
if len(worker_paths) != 2:
    raise RuntimeError(f"Expected two worker checkpoints; found {worker_paths}")
PREDICTIONS_PATH = OUTPUT_DIRECTORY / (
    "screen-predictions.jsonl" if ACTIVE_STAGE is Phase7Stage.SCREEN else "pilot-predictions.jsonl"
)
adapter_predictions = merge_prediction_shards(
    evaluated_selection,
    worker_paths,
    PREDICTIONS_PATH,
)
print(f"Merged predictions: {len(adapter_predictions):,}")
print(f"Prediction checkpoint: {PREDICTIONS_PATH}")

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
print("Inference workers exited; GPU cache cleared before BERTScore.")

In [ ]:
from bert_score import score as bert_score


def load_scores(path):
    if not path.is_file():
        return {}
    return {
        row["record_id"]: float(row["f1"])
        for row in (
            json.loads(line)
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        )
    }


base_bertscore_all = load_scores(BASE_BERTSCORE_PATH)
BERTSCORE_PATH = OUTPUT_DIRECTORY / (
    "screen-bertscore.jsonl" if ACTIVE_STAGE is Phase7Stage.SCREEN else "pilot-bertscore.jsonl"
)
adapter_bertscore = load_scores(BERTSCORE_PATH)
if ACTIVE_STAGE is Phase7Stage.FULL and not adapter_bertscore:
    adapter_bertscore.update(load_scores(OUTPUT_DIRECTORY / "screen-bertscore.jsonl"))
selection_by_id = {example.record_id: example for example in evaluated_selection}
predictions_by_id = {prediction.record_id: prediction for prediction in adapter_predictions}
missing_ids = [
    example.record_id
    for example in evaluated_selection
    if example.record_id not in adapter_bertscore
]
if missing_ids:
    references = [selection_by_id[record_id].reference_summary for record_id in missing_ids]
    candidates = [predictions_by_id[record_id].prediction for record_id in missing_ids]
    _, _, f1 = bert_score(
        candidates,
        references,
        model_type=PHASE5_REPORT.config.bertscore_model_id,
        lang="en",
        batch_size=16,
        device="cuda:0",
        rescale_with_baseline=True,
        verbose=True,
    )
    adapter_bertscore.update(dict(zip(missing_ids, map(float, f1.tolist()), strict=True)))
target_ids = {example.record_id for example in evaluated_selection}
adapter_bertscore = {record_id: adapter_bertscore[record_id] for record_id in target_ids}
temporary = BERTSCORE_PATH.with_name(f"{BERTSCORE_PATH.name}.tmp")
with temporary.open("w", encoding="utf-8", newline="\n") as output:
    for example in evaluated_selection:
        output.write(
            json.dumps({"record_id": example.record_id, "f1": adapter_bertscore[example.record_id]})
        )
        output.write("\n")
temporary.replace(BERTSCORE_PATH)
base_bertscore = {record_id: base_bertscore_all[record_id] for record_id in target_ids}
print(f"BERTScore records added: {len(missing_ids):,}")
print(f"BERTScore coverage: {len(adapter_bertscore):,}")

In [ ]:
from summarizer_ml.evaluation import evaluate_adapter_comparison

result = evaluate_adapter_comparison(
    full_selection,
    evaluated_selection,
    base_predictions,
    adapter_predictions,
    base_bertscore,
    adapter_bertscore,
    OUTPUT_DIRECTORY,
    CONFIG,
    ACTIVE_STAGE,
    max_new_tokens=PHASE5_REPORT.config.max_new_tokens,
    benchmark_selection_path=SELECTION_PATH,
    base_predictions_path=BASE_PREDICTIONS_PATH,
    adapter_path=ADAPTER_PATH,
    adapter_predictions_path=PREDICTIONS_PATH,
)
print(result.report.model_dump_json(indent=2))
if result.report.decision is Phase7Decision.CONTINUE:
    print("Screen gate passed. Change RUN_STAGE to 'full' and rerun from the configuration cell.")
elif result.report.decision is Phase7Decision.STOP:
    print("Screen gate stopped evaluation. Do not run the remaining 400 records.")
elif result.report.decision is Phase7Decision.ACCEPT_PILOT:
    print("Full benchmark accepted the pilot adapter as the final fine-tuned candidate.")
else:
    print("Full benchmark rejected the pilot adapter. Do not perform more training automatically.")

In [ ]:
required = [
    OUTPUT_DIRECTORY / "phase7-config.json",
    OUTPUT_DIRECTORY / "workers" / "rank-00.jsonl",
    OUTPUT_DIRECTORY / "workers" / "rank-01.jsonl",
    PREDICTIONS_PATH,
    BERTSCORE_PATH,
    OUTPUT_DIRECTORY / f"{RUN_STAGE}-comparison-report.json",
]
for path in required:
    print(
        path.relative_to(OUTPUT_DIRECTORY),
        "exists:",
        path.is_file(),
        "bytes:",
        path.stat().st_size if path.exists() else 0,
    )
    if not path.is_file():
        raise RuntimeError(f"Missing Phase 7 artifact: {path}")
print(f"Phase 7 {RUN_STAGE} artifact gate passed with decision: {result.report.decision.value}")